# TechStream Anomaly Detection System
## Detección de Anomalías en Servidores con Redes Neuronales
**Autor:** [Tu nombre]  |  **Fecha:** 24 de abril de 2026  |  **Framework:** PyTorch

> Sistema de clasificación binaria para detectar fallos en telemetría de servidores
> en tiempo real, minimizando el impacto operacional no detectado.


---

## 1. Setup e Imports

Configuración del entorno y carga de módulos del proyecto.


In [ ]:
# Raíz del repo (notebooks/ → subir un nivel) y rutas reproducibles
from __future__ import annotations

import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

%matplotlib inline

# UTF-8 en Windows para emojis en salidas
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except (OSError, ValueError):
        pass

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_CSV = PROJECT_ROOT / "data" / "sensors.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "model.pth"
HISTORY_JSON = PROJECT_ROOT / "models" / "training_history.json"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Paleta TechStream (normal / fallo)
C_NORMAL = "#2196F3"
C_FAIL = "#F44336"

sns.set_theme(style="whitegrid", context="notebook")

SEED_NB = 42
random.seed(SEED_NB)
np.random.seed(SEED_NB)
torch.manual_seed(SEED_NB)


---

## 2. Dataset Overview

Telemetría sintética de servidores (`data/sensors.csv`): siete variables continuas y la etiqueta binaria `failure` (0 = operación normal, 1 = fallo). Los datos se generan con reglas físicas (sobrecarga, térmica, errores, latencia) y una mezcla 80/20 entre rangos normales y elevados, con barajado reproducible.


In [ ]:
from IPython.display import display

df = pd.read_csv(DATA_CSV)
FEATURE_COLS = [
    "cpu_usage",
    "memory_usage",
    "temperature",
    "network_traffic",
    "disk_io",
    "error_rate",
    "response_time",
]
TARGET = "failure"

display(df.head(10))
df.info()
display(df.describe())

counts = df[TARGET].value_counts().sort_index()
pct = counts / len(df) * 100
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(counts.index.astype(str), counts.values, color=[C_NORMAL, C_FAIL])
ax.set_xlabel("failure")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de clases")
for i, (k, v) in enumerate(counts.items()):
    ax.text(i, v, f"{v}\n({pct[k]:.1f}%)", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()


---

## 3. Análisis Exploratorio (EDA)

Análisis de distribuciones y correlaciones entre features.


In [ ]:
# 1) Histogramas por feature y clase
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.ravel()
for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    df.loc[df[TARGET] == 0, col].hist(ax=ax, bins=30, alpha=0.55, color=C_NORMAL, label="failure=0")
    df.loc[df[TARGET] == 1, col].hist(ax=ax, bins=30, alpha=0.55, color=C_FAIL, label="failure=1")
    ax.set_title(col)
    ax.legend(fontsize=8)
for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

# 2) Heatmap de correlaciones (features + target)
fig, ax = plt.subplots(figsize=(12, 6))
corr = df[FEATURE_COLS + [TARGET]].corr()
sns.heatmap(corr, ax=ax, cmap="coolwarm", center=0)
ax.set_title("Correlaciones (Pearson)")
plt.tight_layout()
plt.show()

# 3) Boxplots por feature y clase
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.ravel()
for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    sns.boxplot(
        data=df,
        x=TARGET,
        y=col,
        ax=ax,
        hue=TARGET,
        palette=[C_NORMAL, C_FAIL],
        dodge=False,
        legend=False,
    )
    ax.set_title(col)
for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

# 4) Pairplot: 3 features más correlacionadas con failure (excl. la diagonal target)
corr_fail = df[FEATURE_COLS + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
top3 = list(corr_fail.index[:3])
plot_df = df[top3 + [TARGET]].copy()
plot_df[TARGET] = plot_df[TARGET].astype(int)
g = sns.pairplot(
    plot_df,
    hue=TARGET,
    corner=True,
    palette={0: C_NORMAL, 1: C_FAIL},
    plot_kws={"alpha": 0.35, "s": 18, "edgecolor": "none"},
    height=3.2,
)
g.fig.suptitle("Pairplot — top 3 features vs failure", y=1.02)
plt.tight_layout()
plt.show()


---

## 4. Preprocesamiento

Estrategia: partición estratificada 70/15/15 (train/val/test), estandarización con `StandardScaler` ajustada **solo** en train, y manejo del desbalance con `WeightedRandomSampler` y `pos_weight` en entrenamiento (ver `src/dataset.py`, `src/train.py`).


In [ ]:
from IPython.display import display

from src.dataset import FEATURE_COLS as DS_FEATURES
from src.dataset import SEED, TEST_SIZE, VAL_SIZE, load_data

bundle = load_data(DATA_CSV, TEST_SIZE, VAL_SIZE, SEED)

for name, X, y in [
    ("Train", bundle.X_train, bundle.y_train),
    ("Val", bundle.X_val, bundle.y_val),
    ("Test", bundle.X_test, bundle.y_test),
]:
    y = np.asarray(y).ravel()
    print(f"{name}: X={X.shape}  y={y.shape}  fallos={100 * (y == 1).mean():.1f}%")

scaler = bundle.scaler
mean_df = pd.DataFrame([scaler.mean_], columns=DS_FEATURES)
scale_df = pd.DataFrame([scaler.scale_], columns=DS_FEATURES)
print("\nMedia por feature (train fit):")
display(mean_df)
print("Escala (std) por feature:")
display(scale_df)


---

## 5. Arquitectura del Modelo

Se usa un **MLP** (perceptrón multicapa) porque cada fila es un **snapshot** instantáneo de sensores: no hay secuencia temporal en el tensor de entrada. Una **LSTM** tendría sentido si alimentáramos ventanas temporales (`[t-k, …, t]`) por servidor.

**BatchNorm1d** estabiliza la escala de activaciones por mini-lote y acelera la convergencia; **Dropout** actúa como regularizador frente al sobreajuste en presencia de clases desbalanceadas.


In [ ]:
from src.model import build_model

model = build_model(input_dim=len(FEATURE_COLS), hidden_dims=[128, 64, 32], dropout_rate=0.3)
print("Parámetros entrenables:", model.get_num_params())


### Diagrama ASCII (flujo de tensores)

```
Input (batch, 7)
    │
    ▼
┌───────────────────────────────┐
│ Linear(7→128) + BN + ReLU + DO│
└───────────────────────────────┘
    │
    ▼
┌────────────────────────────────┐
│ Linear(128→64) + BN + ReLU + DO│
└────────────────────────────────┘
    │
    ▼
┌────────────────────────────────┐
│ Linear(64→32) + BN + ReLU + DO │
└────────────────────────────────┘
    │
    ▼
 Linear(32→1)  → logits (batch,)   [sin sigmoid; BCEWithLogitsLoss en train]
```


---

## 6. Entrenamiento

Hiperparámetros alineados con `src/train.py`: Adam `lr=1e-3`, batch 64, hasta 150 épocas con **early stopping** (paciencia 10), `ReduceLROnPlateau` (×0.5, paciencia 5) y pérdida **BCEWithLogitsLoss** con `pos_weight` por desbalance.


In [ ]:
from IPython.display import display

from src.evaluate import plot_threshold_analysis, plot_training_history, training_summary_table

fig = plot_training_history(HISTORY_JSON, figsize=(12, 6))
plt.show()

fig2 = plot_threshold_analysis(figsize=(12, 6))
plt.show()

summary = training_summary_table()
display(summary)


---

## 7. Evaluación en Test Set

Evaluación final sobre datos no vistos durante el entrenamiento (split estratificado reproducible).


In [ ]:
from IPython.display import Markdown, display

from src.evaluate import (
    compute_metrics,
    plot_confusion_matrix_fig,
    plot_pr_curve_fig,
    plot_roc_curve_fig,
    plot_score_distribution_fig,
)

metrics_df = compute_metrics(0.5)

plot_confusion_matrix_fig(figsize=(12, 6))
plt.show()

plot_roc_curve_fig(figsize=(12, 6))
plt.show()

plot_pr_curve_fig(figsize=(12, 6))
plt.show()

plot_score_distribution_fig(figsize=(12, 6))
plt.show()

recall_val = float(metrics_df.loc["recall", "valor"])
display(
    Markdown(
        f"**Recall en test:** {recall_val:.3f}. "
        "En detección de fallos, un recall alto reduce **falsos negativos** "
        "(incidentes no alertados), priorizando la cobertura de anomalías."
    )
)


---

## 8. Inferencia en Tiempo Real

Ejemplo de uso del modelo con un registro nuevo (preprocesado con el mismo `StandardScaler` del entrenamiento).


In [ ]:
from src.dataset import load_data, SEED, TEST_SIZE, VAL_SIZE
from src.model import AnomalyDetector

# Simular un servidor en estado crítico (reglas de negocio del generador)
new_reading = {
    "cpu_usage": 95.0,
    "memory_usage": 88.0,
    "temperature": 72.0,
    "network_traffic": 450.0,
    "disk_io": 180.0,
    "error_rate": 9.5,
    "response_time": 850.0,
}

bundle_inf = load_data(DATA_CSV, TEST_SIZE, VAL_SIZE, SEED)
scaler_inf = bundle_inf.scaler

row = np.array([[new_reading[c] for c in FEATURE_COLS]], dtype=np.float64)
x_scaled = scaler_inf.transform(row).astype(np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = AnomalyDetector(input_dim=len(FEATURE_COLS), hidden_dims=[128, 64, 32], dropout_rate=0.3)
try:
    state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
except TypeError:
    state = torch.load(MODEL_PATH, map_location=device)
net.load_state_dict(state)
net.to(device)
net.eval()

with torch.no_grad():
    logit = net(torch.from_numpy(x_scaled).to(device))
    prob = torch.sigmoid(logit).item()

print(f"Probabilidad de fallo: {prob:.4f}")
if prob >= 0.5:
    print("🚨 ALERTA: Fallo detectado")
else:
    print("✅ Normal")


---

## 9. Conclusiones y Próximos Pasos

### Resultados obtenidos

| Métrica (test, umbral 0.5) | Valor |
|----------------------------|-------|
| Ver tabla impresa en la Sección 7 (`compute_metrics`) | — |

El pipeline cubre generación de datos, entrenamiento con control de desbalance, early stopping y evaluación con curvas ROC/PR.

### Mejoras propuestas

- **LSTM / Transformer** para capturar dependencias temporales en ventanas por servidor.
- **Umbral dinámico** (p. ej. por percentil o coste FN/FP) en lugar de 0.5 fijo.
- **Sistema de alertas en streaming** (Kafka / MQTT) con latencia sub-segundo.
- **Reentrenamiento continuo** con datos nuevos etiquetados y vigilancia de deriva.


In [ ]:
print("✅ Notebook ejecutado correctamente")
